# W4a · 无损 PNG 对照与 PS 分段计时
复用已板测 W3 服务及 W2 bit/HWH；只改 PS 编码设置并补分段计时。无 HDMI。
上传本 Notebook 和 browser_w4a_png_r1.zip 到新 retina_browser_w4a_r1 文件夹。
只在上一步结束且无报错后运行下一框；不要 Run All。FAULT 时保留内核和缓冲，不重复加载。


## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [1]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('browser_w4a_png_r1.zip')
expected_bundle_sha256 = '5a782132d4e350eb2392451ed7693abfdfa512db81b2c466cd7e5c22bed807ef'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('browser_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'browser_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 100 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'browser_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'browser_w4a_png_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: paired W2 and browser application bytes checked; FPGA untouched')


PACKAGE_BYTES_PASS: paired W2 and browser application bytes checked; FPGA untouched


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [2]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from pynq_w4a.launch import prepare, start, export_session, finish
browser_context = prepare(browser_package_root, port=8090)


BROWSER_PRECHECK_PASS: paired W2 hashes/runtime/metadata checked; FPGA unchanged
Runtime: {'numpy': '1.21.5', 'Pillow': '9.0.1', 'tornado': '6.1', 'pynq': '3.0.1', 'python': '3.10.4', 'kernel': '5.15.19-xilinx-v2022.1', 'machine': 'armv7l', 'board_clock_utc_unverified': '2022-10-24T07:10:51.106168+00:00', 'runtime_driver_sources': {'DMA': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/lib/dma.py', 'bytes': 26228, 'sha256': 'a7e92fd180d9fa3322e81b49bb4141e08c8f5c8cb685fe608793b46acfe11d49'}, 'AxiGPIO': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/lib/axigpio.py', 'bytes': 10935, 'sha256': 'bb386977825c5c36ee4e5ed814331c1f3105858135fc581362d01e9955d21f27'}, 'allocate': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/buffer.py', 'bytes': 4676, 'sha256': 'c3cfea4e2a94aa9b1981ae79210a69eee0f17b5a5c4cd26e7bc2cf7de01fb236'}, 'Overlay': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlay.py', 'b

## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [3]:
browser_start_status = await start(browser_context)
print(browser_start_status)


BROWSER_SERVICE_READY: W2 A/C2 loaded once; no frame has been sent yet
Open http://192.168.2.99:8090/
Session: 8f2289bdbdc94093b63dbfeb9eeb4d5c
{'session_id': '8f2289bdbdc94093b63dbfeb9eeb4d5c', 'service_state': 'READY', 'hardware_state': 'IDLE', 'overlay_downloads': 1, 'last_completed_frame_status': 0, 'active_run_id': None, 'pending': 0, 'completed': 0, 'accepted': 0, 'run_limit': 200, 'retained_buffer_count': 0, 'error': None, 'evidence_kind': 'PYNQ_PL_DMA', 'board_acceptance': 'PENDING_INDEPENDENT_VERIFICATION'}


## 4. 浏览器对照实验
打开 http://192.168.2.99:8090/，确认页眉W4a。每次完成再继续，实际算法/阈值回读应一致。
固定 im0001 /512/阈值28/最终掩膜，依次手动运行：
1. A + 原方案；2. A + 快速候选；3. C2 + 原方案；4. C2 + 快速候选。
同算法两种传输设置的图像和Dice应一致。PNG文件字节/大小可不同，解码像素须一致。
然后固定 im0001 /512/C2/28/最终掩膜：
先选“原方案 · 默认 PNG”，点击“预热3帧+测量20帧”；完成并截图后，
再选“快速候选 · 无损 PNG”，点击同一按钮；完成并截图。
两组保持网页前台，每组3个新PL预热帧+20个计时帧，全部保存，FPS只计后20帧。
本轮共50个新PL任务，一次加载；不是正式3预热+100帧×三配置×三复测，也不是Q3全数据集实验。
比较PS解包、PIL转换、PNG编码/写入、哈希、文件大小和浏览器端到端耗时。没有预定提速结论。
返回Notebook运行状态框，应IDLE、completed至少50、pending0、无错误和保留缓冲。


In [4]:
print(browser_context['manager'].status())


{'session_id': '8f2289bdbdc94093b63dbfeb9eeb4d5c', 'service_state': 'READY', 'hardware_state': 'IDLE', 'overlay_downloads': 1, 'last_completed_frame_status': 819250, 'active_run_id': None, 'pending': 0, 'completed': 50, 'accepted': 50, 'run_limit': 200, 'retained_buffer_count': 0, 'error': None, 'evidence_kind': 'PYNQ_PL_DMA', 'board_acceptance': 'PENDING_INDEPENDENT_VERIFICATION'}


## 5. 打包本次真实记录
须无在途/待执行任务。应打印 BROWSER_RECORDS_READY。完整会话包含每任务原packed NPY、PNG、JSON、CSV及浏览器绘制观测；尚待项目独立复核，不自动宣称验收通过。

In [5]:
import shutil
browser_export_path = await export_session(browser_context)
shutil.copyfile(browser_export_path, Path.cwd() / 'browser_w4a_session_bundle.zip')
print('DOWNLOAD: browser_w4a_session_bundle.zip in this notebook folder')


BROWSER_RECORDS_READY: /home/xilinx/jupyter_notebooks/retina_browser_w4a_r1/browser_w4a_png_r1/browser_results/8f2289bdbdc94093b63dbfeb9eeb4d5c/exports/browser_session_8f2289bdbdc94093b63dbfeb9eeb4d5c.zip
DOWNLOAD: browser_w4a_session_bundle.zip in this notebook folder


## 6. 安全结束并下载
仅READY/IDLE且无在途或故障缓冲时关闭HTTP服务。FAULT时本框拒绝关闭，保留内核和原记录并反馈。
完成后Ctrl+S；File → Download as → Notebook下载执行记录。回文件列表下载 `browser_w4a_session_bundle.zip`。反馈这两个文件和浏览器截图。

In [6]:
await finish(browser_context)


BROWSER_SERVICE_CLOSED_IDLE: native buffers already released; original records retained
